In [1]:
import numpy as np
import pandas as pd
import folium

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
places = pd.read_csv("../data/processed/manali_hybrid_scores.csv")

print("Places loaded:", len(places))
places.head()


Places loaded: 20


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,...,interest_count,structured_score,recommendation_text,tfidf_score,semantic_text,semantic_score,rating_score,log_reviews,popularity_score,hybrid_score
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,...,6,0.600099,Hadimba Devi Temple Tourist attraction culture...,0.0,Hadimba Devi Temple. Tourist attraction. cultu...,0.321686,0.777778,10.813539,1.000000,0.463197
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,...,5,0.717137,Old Manali snow point Tourist attraction cultu...,0.0,Old Manali snow point. Tourist attraction. cul...,0.374756,0.777778,6.061457,0.429428,0.451321
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,...,2,0.472456,"Nehru Kund Tourist attraction history, photogr...",0.0,"Nehru Kund. Tourist attraction. history, photo...",0.417762,0.555556,8.957768,0.777182,0.404494
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,...,0,0.000000,Kullu Manali River rafting Tourist attraction,0.0,Kullu Manali River rafting. Tourist attraction.,0.315588,0.666667,4.488636,0.240583,0.218735
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,...,4,0.734968,"Jogini Falls Tourist attraction family, histor...",0.0,"Jogini Falls. Tourist attraction. family, hist...",0.418287,0.777778,9.291275,0.817225,0.507617


In [3]:
itinerary_df = pd.read_csv(
    "../data/processed/manali_itinerary_baseline.csv"
)

print("Itinerary rows:", len(itinerary_df))
itinerary_df


Itinerary rows: 10


,day,stop,place,arrival,departure,travel_before_minutes,visit_minutes,hybrid_score
0,1,1,Jogini Falls,9:00 AM,10:30 AM,0.0,90,0.507617
1,1,2,Nehru Kund,10:33 AM,11:33 AM,3.5,60,0.404494
2,1,3,Hadimba Devi Temple,11:43 AM,12:43 PM,10.0,60,0.463197
3,1,4,Old Manali snow point,12:43 PM,2:13 PM,0.4,90,0.451321
4,1,5,Lama Dugh Trek Start Point,2:15 PM,4:45 PM,1.1,150,0.423811
5,2,1,Manali View Point,9:00 AM,9:45 AM,0.0,45,0.468197
6,2,2,Van Vihar National Park,9:46 AM,11:46 AM,1.5,120,0.447468
7,2,3,Kharma valley,11:52 AM,12:52 PM,6.3,60,0.406514
8,2,4,Baror Parsha Waterfall,1:06 PM,2:36 PM,13.4,90,0.397755
9,2,5,Gulaba Viewpoint,3:05 PM,3:50 PM,29.7,45,0.397028


In [4]:
map_df = itinerary_df.merge(
    places[[
        "name",
        "latitude",
        "longitude",
        "rating",
        "reviews",
        "hybrid_score"
    ]],
    left_on="place",
    right_on="name",
    how="left"
)

map_df[[
    "day",
    "stop",
    "place",
    "latitude",
    "longitude"
]]


,day,stop,place,latitude,longitude
0,1,1,Jogini Falls,32.275076,77.188146
1,1,2,Nehru Kund,32.285982,77.179824
2,1,3,Hadimba Devi Temple,32.248353,77.181573
3,1,4,Old Manali snow point,32.249112,77.180076
4,1,5,Lama Dugh Trek Start Point,32.248903,77.175013
5,2,1,Manali View Point,32.233835,77.187359
6,2,2,Van Vihar National Park,32.239113,77.189089
7,2,3,Kharma valley,32.254802,77.168016
8,2,4,Baror Parsha Waterfall,32.206785,77.184900
9,2,5,Gulaba Viewpoint,32.317903,77.189340


In [5]:
missing_coordinates = map_df[
    map_df[["latitude", "longitude"]].isnull().any(axis=1)
]

print("Places with missing coordinates:", len(missing_coordinates))

if not missing_coordinates.empty:
    display(
        missing_coordinates[[
            "place",
            "latitude",
            "longitude"
        ]]
    )


Places with missing coordinates: 0


In [6]:
center_lat = map_df["latitude"].mean()
center_lon = map_df["longitude"].mean()

print("Map center:", center_lat, center_lon)


Map center: 32.25598632 77.18233357


In [7]:
travel_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13
)

travel_map


In [8]:
for _, row in map_df.dropna(
    subset=["latitude", "longitude"]
).iterrows():

    popup_text = (
        f"<b>Day {int(row['day'])}, Stop {int(row['stop'])}</b><br>"
        f"<b>{row['place']}</b><br>"
        f"Arrival: {row['arrival']}<br>"
        f"Departure: {row['departure']}<br>"
        f"Rating: {row['rating']}<br>"
        f"Reviews: {int(row['reviews'])}"
    )

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=(
            f"Day {int(row['day'])} - "
            f"Stop {int(row['stop'])}: {row['place']}"
        )
    ).add_to(travel_map)


In [9]:
for day in sorted(map_df["day"].unique()):

    day_data = map_df[
        map_df["day"] == day
    ].sort_values("stop")

    coordinates = (
        day_data[["latitude", "longitude"]]
        .dropna()
        .values
        .tolist()
    )

    if len(coordinates) >= 2:
        folium.PolyLine(
            coordinates,
            tooltip=f"Day {int(day)} route",
            weight=4
        ).add_to(travel_map)


In [10]:
travel_map


In [11]:
output_path = "../data/processed/manali_itinerary_map.html"

travel_map.save(output_path)

print(f"✅ Interactive map saved to: {output_path}")


✅ Interactive map saved to: ../data/processed/manali_itinerary_map.html


In [12]:
daily_summary = (
    map_df.sort_values(["day", "stop"])
    .groupby("day")
    .agg(
        first_stop=("place", "first"),
        last_stop=("place", "last"),
        stops=("place", "count")
    )
    .reset_index()
)

daily_summary


,day,first_stop,last_stop,stops
0,1,Jogini Falls,Lama Dugh Trek Start Point,5
1,2,Manali View Point,Gulaba Viewpoint,5


# 🎯 What we achieved

```text
Hybrid Recommendation
        ↓
Itinerary Optimizer
        ↓
Day-by-Day Schedule
        ↓
Interactive Map 🗺️
```

The next stage is to improve realism by enriching visit duration, opening hours, travel information and other trip constraints, then connect the engine to a FastAPI backend and Streamlit interface.
